In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [9]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3660498261451721
epoch:  1, loss: 0.046265941113233566
epoch:  2, loss: 0.030165085569024086
epoch:  3, loss: 0.01554983388632536
epoch:  4, loss: 0.014342608861625195
epoch:  5, loss: 0.013783163391053677
epoch:  6, loss: 0.013444735668599606
epoch:  7, loss: 0.012956508435308933
epoch:  8, loss: 0.012287912890315056
epoch:  9, loss: 0.011938954703509808
epoch:  10, loss: 0.011740680783987045
epoch:  11, loss: 0.011405359022319317
epoch:  12, loss: 0.010962115600705147
epoch:  13, loss: 0.010768111795186996
epoch:  14, loss: 0.010665993206202984
epoch:  15, loss: 0.010399134829640388
epoch:  16, loss: 0.01018644031137228
epoch:  17, loss: 0.010095431469380856
epoch:  18, loss: 0.010068792849779129
epoch:  19, loss: 0.009818035177886486
epoch:  20, loss: 0.009726949967443943
epoch:  21, loss: 0.009684208780527115
epoch:  22, loss: 0.009561827406287193
epoch:  23, loss: 0.009467361494898796
epoch:  24, loss: 0.009429126977920532
epoch:  25, loss: 0.009375534020364285


In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.8912698030471802
Test metrics:  R2 = 0.9074886441230774


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LevenbergMarquardtLS(model=model,batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.44608694314956665
epoch:  1, loss: 0.20099163055419922
epoch:  2, loss: 0.13603578507900238
epoch:  3, loss: 0.08304242044687271
epoch:  4, loss: 0.04262040555477142
epoch:  5, loss: 0.02834988944232464
epoch:  6, loss: 0.02649841457605362
epoch:  7, loss: 0.02439531311392784
epoch:  8, loss: 0.022682951763272285
epoch:  9, loss: 0.02143014781177044
epoch:  10, loss: 0.0202310960739851
epoch:  11, loss: 0.020150894299149513
epoch:  12, loss: 0.01877858303487301
epoch:  13, loss: 0.017798351123929024
epoch:  14, loss: 0.017102966085076332
epoch:  15, loss: 0.016754932701587677
epoch:  16, loss: 0.015506736934185028
epoch:  17, loss: 0.014741170220077038
epoch:  18, loss: 0.014263082295656204
epoch:  19, loss: 0.014052467420697212
epoch:  20, loss: 0.013061570934951305
epoch:  21, loss: 0.012510538101196289
epoch:  22, loss: 0.012198043055832386
epoch:  23, loss: 0.012186417356133461
epoch:  24, loss: 0.011391839943826199
epoch:  25, loss: 0.011032447218894958
epoch:  

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9555864930152893
Test metrics:  R2 = 0.9559220671653748
